In [110]:
# Import the necessary packages
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Scikit-learn tools for modeling
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [111]:
retail_sales = pd.read_csv('train.csv')
retail_sales_test = pd.read_csv('test.csv')
retail_sales.head()

,id,category,quantity,total_spent,payment_method,location,transaction_date,discount_applied
0,TXN_6867343,Patisserie,10,185.0,Digital Wallet,Online,4/8/2024,TRUE
1,TXN_3731986,Milk Products,9,261.0,Digital Wallet,Online,7/23/2023,TRUE
2,TXN_9303719,Butchers,2,43.0,Credit Card,Online,10/5/2022,FALSE
3,TXN_9458126,Beverages,9,247.5,Credit Card,Online,5/7/2022,Unknown
4,TXN_4575373,Food,7,87.5,Digital Wallet,Online,10/2/2022,FALSE


# Feature Engineering Techniques

In [113]:
# ONE-HOT ENCODE the "category" column
category_dummies = pd.get_dummies(retail_sales['category'])
retail_sales = pd.concat([retail_sales, category_dummies], axis=1)

category_dummies = pd.get_dummies(retail_sales_test['category'])
retail_sales_test = pd.concat([retail_sales_test, category_dummies], axis=1)


In [114]:
# ONE-HOT ENCODE the "payment_method" column
payment_dummies = pd.get_dummies(retail_sales['payment_method'], drop_first=True)
retail_sales = pd.concat([retail_sales, payment_dummies], axis=1)

payment_dummies = pd.get_dummies(retail_sales_test['payment_method'], drop_first= True)
retail_sales_test = pd.concat([retail_sales_test, payment_dummies], axis=1)


In [115]:
# ONE-HOT ENCODE the "location" column
location_dummies = pd.get_dummies(retail_sales['location'], drop_first= True)
retail_sales = pd.concat([retail_sales, location_dummies], axis=1)

location_dummies = pd.get_dummies(retail_sales_test['location'], drop_first= True)
retail_sales_test = pd.concat([retail_sales_test, location_dummies], axis=1)


In [116]:
# ONE-HOT ENCODE the "discount_applied" column
discount_dummies = pd.get_dummies(retail_sales['discount_applied'], drop_first= True)
retail_sales = pd.concat([retail_sales, discount_dummies], axis=1)

discount_dummies = pd.get_dummies(retail_sales_test['discount_applied'], drop_first= True)
retail_sales_test = pd.concat([retail_sales_test, discount_dummies], axis=1)


In [117]:
# SPLIT "transaction_date" into "day", "month", "year", and "dayofweek" variables
# "transaction_date" must first be CONVERTED TO DATETIME type
retail_sales['transaction_date'] = pd.to_datetime(retail_sales['transaction_date'])

retail_sales['day'] = retail_sales['transaction_date'].dt.day
retail_sales['month'] = retail_sales['transaction_date'].dt.month
retail_sales['year'] = retail_sales['transaction_date'].dt.year
retail_sales['dayofweek'] = retail_sales['transaction_date'].dt.dayofweek

retail_sales_test['transaction_date'] = pd.to_datetime(retail_sales_test['transaction_date'])

retail_sales_test['day'] = retail_sales_test['transaction_date'].dt.day
retail_sales_test['month'] = retail_sales_test['transaction_date'].dt.month
retail_sales_test['year'] = retail_sales_test['transaction_date'].dt.year
retail_sales_test['dayofweek'] = retail_sales_test['transaction_date'].dt.dayofweek


In [118]:
# Create variables "[category]_quantity" (such as "Food_quantity" or "Patisserie_quantity")
# which multiplies one-hot encoded category by quantity
# USE THE ONE-HOT ENCODED CATEGORY COLUMNS WE CREATED
retail_sales["beverage_quantity"] = retail_sales["Beverages"] * retail_sales["quantity"]
retail_sales["butcher_quantity"] = retail_sales["Butchers"] * retail_sales["quantity"]
retail_sales["computer_quantity"] = retail_sales["Computers and electric accessories"] * retail_sales["quantity"]
retail_sales["electric_house_quantity"] = retail_sales["Electric household essentials"] * retail_sales["quantity"]
retail_sales["food_quantity"] = retail_sales["Food"] * retail_sales["quantity"]
retail_sales["furniture_quantity"] = retail_sales["Furniture"] * retail_sales["quantity"]
retail_sales["milk_quantity"] = retail_sales["Milk Products"] * retail_sales["quantity"]
retail_sales["patisserie_quantity"] = retail_sales["Patisserie"] * retail_sales["quantity"]

retail_sales_test["beverage_quantity"] = retail_sales_test["Beverages"] * retail_sales_test["quantity"]
retail_sales_test["butcher_quantity"] = retail_sales_test["Butchers"] * retail_sales_test["quantity"]
retail_sales_test["computer_quantity"] = retail_sales_test["Computers and electric accessories"] * retail_sales_test["quantity"]
retail_sales_test["electric_house_quantity"] = retail_sales_test["Electric household essentials"] * retail_sales_test["quantity"]
retail_sales_test["food_quantity"] = retail_sales_test["Food"] * retail_sales_test["quantity"]
retail_sales_test["furniture_quantity"] = retail_sales_test["Furniture"] * retail_sales_test["quantity"]
retail_sales_test["milk_quantity"] = retail_sales_test["Milk Products"] * retail_sales_test["quantity"]
retail_sales_test["patisserie_quantity"] = retail_sales_test["Patisserie"] * retail_sales_test["quantity"]

In [119]:
#select non-redundant features for modeling

features = ["TRUE", "Unknown", "Credit Card", "Digital Wallet", "Online", "day",
            "month", "year", "dayofweek", "beverage_quantity", "butcher_quantity",
            "computer_quantity", "electric_house_quantity", "food_quantity", "furniture_quantity"]

X_train = retail_sales[features]
y_train = retail_sales["total_spent"]
X_test = retail_sales_test[features]
X_train.head()

,TRUE,Unknown,Credit Card,Digital Wallet,Online,day,month,year,dayofweek,beverage_quantity,butcher_quantity,computer_quantity,electric_house_quantity,food_quantity,furniture_quantity
0,True,False,False,True,True,8,4,2024,0,0,0,0,0,0,0
1,True,False,False,True,True,23,7,2023,6,0,0,0,0,0,0
2,False,False,True,False,True,5,10,2022,2,0,2,0,0,0,0
3,False,True,True,False,True,7,5,2022,5,9,0,0,0,0,0
4,False,False,False,True,True,2,10,2022,6,0,0,0,0,7,0


In [120]:
#Use this cell if you want to do further internal testing on your own with the train set only, you can skip for now

#X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

### There will be fill-in-the-blanks represented by "____". The tasks will be described in the green comments above the code. If you are struggling, raise your hand or look back at the mock notebook. You only have to use one of these models for the Kaggle competition submission (or one of your own).

In [121]:
#linear regression
lin_reg = LinearRegression()
#Train the model
lin_reg.___(X_train, y_train)
total_spent = lin_reg.predict(X_test)

AttributeError: 'LinearRegression' object has no attribute '___'

In [ ]:
#ridge regression
scaler = StandardScaler()

# fit and transform on the training data, transform only on the test data
X_train_scaled = scaler.________(X_train)
X_test_scaled = scaler.transform(X_test)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
total_spent = ridge.predict(X_test_scaled)

In [ ]:
#random forest
rf = RandomForestRegressor(n_estimators=___, max_depth=___)
rf.fit(X_train, y_train)
total_spent = rf.predict(X_test)

In [ ]:
# Convert the array to a DataFrame
dataframe = pd.DataFrame(total_spent)
dataframe.columns = ['total_spent']

# Save the DataFrame to a CSV file
dataframe.to_csv('total_spent.csv', index=False)

### You can now make a submission to the Kaggle competition!! download the total_spent.csv and upload it to the competition to be entered!